# 3. Stationarity and Cyclostationarity Assessment


## Load the derived residual series

This notebook uses the residual series created by `02_decomposition.ipynb`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

residual_series = pd.read_csv(
    "data/residual_series.csv",
    parse_dates=["date"],
    index_col="date"
)["residual"].dropna()

print(f"Loaded {len(residual_series)} residual observations.")


## Concepts

**Stationarity** asks whether important statistical characteristics remain reasonably stable over time. **Cyclostationarity** asks whether statistical behavior changes in a structured periodic way.

### Analogy
A washing machine can have broadly stable behavior while its vibration changes at repeated points in each cycle. That repeated change is an intuition for periodic structure.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

# === 1. Codifference Function ===
def codifference(x, lag=1):
    return x[lag:] - x[:-lag]

# === 2. Rolling MAD for Codiff Stability ===
def rolling_codiff_mad(series, window=100, lag=1):
    mad_vals = []
    for i in range(len(series) - window):
        chunk = series[i:i+window]
        diff = codifference(chunk, lag)
        mad_vals.append(np.median(np.abs(diff - np.median(diff))))
    return mad_vals

# === 3. ADoF Test for Cyclostationarity ===
def adof_test(series, period):
    phases = [[] for _ in range(period)]
    for t in range(len(series)):
        phases[t % period].append(series[t])
    medians = [np.median(phase) for phase in phases]
    variance = np.var(medians)
    return variance, medians

# === 4. Apply to Residual Series ===
y = residual_series.values
index = residual_series.index
y_diff = codifference(y, lag=1)

# === 5. ADF Test ===
print("\n--- Stationarity Check ---")
try:
    adf_stat, p_val, *_ = adfuller(y_diff)
    print(f"ADF Statistic: {adf_stat:.4f}")
    print(f"p-value: {p_val:.4f}")
    adf_result = p_val < 0.05
    print("Result: Stationary" if adf_result else "Result: Not Stationary")
except Exception as e:
    print("ADF Test failed (likely infinite variance):", e)
    adf_result = None

# === 6. Rolling MAD Plot ===
mad_vals = rolling_codiff_mad(y, window=100, lag=1)

plt.figure(figsize=(10, 4))
plt.plot(mad_vals, label="Rolling MAD")
plt.title("Rolling MAD of 1st-Order Codifferenced Residuals")
plt.xlabel("Window Index")
plt.ylabel("Median Absolute Deviation")
plt.grid(True)
plt.tight_layout()
plt.show()

# Infer from MAD plot: if mostly flat → stationary
mad_std = np.std(mad_vals)
mad_mean = np.mean(mad_vals)
stationary_from_mad = mad_std / mad_mean < 0.3  # empirical threshold

# === 7. ADoF Test for Daily Periodicity ===
print("\n--- Cyclostationarity Check ---")
period = 7
adof_var, phase_medians = adof_test(y_diff, period)

print(f"Variance of Phase Medians (ADoF): {adof_var:.6f}")
print(f"Phase Medians: {np.round(phase_medians, 3)}")

plt.figure(figsize=(6, 3))
plt.bar(range(period), phase_medians, color='coral')
plt.title("Phase-wise Medians (Weekly Cyclostationarity Check)")
plt.xlabel("Day of Week (Phase)")
plt.ylabel("Median of Codiff Residuals")
plt.grid(True)
plt.tight_layout()
plt.show()

cyclostationary = adof_var > 1e-3  # empirical threshold

# === 8. Final Summary ===
print("\n=== Final Summary ===")
if adf_result is not None:
    print(f"Stationary (ADF): {'Yes' if adf_result else 'No'}")
else:
    print(f"Stationary (MAD-based): {'Yes' if stationary_from_mad else 'No'}")
print(f"Cyclostationary (ADoF Test): {'Yes' if cyclostationary else 'No'}")



